# Курсовая работа по методам машинного обучения

## Прогнозирование принадлежности профиля к сегменту `United States`

Этот ноутбук является исполняемой версией курсовой работы. Он использует функции из `coursework_ml.py`, чтобы код в ноутбуке и код генерации отчета не расходились.

Задача: по открытым признакам профиля пользователя социальной сети определить, относится ли профиль к классу `United States`.

Важно: `Country` используется только для формирования целевой переменной и не передается в модель как входной признак.

## 1. Импорт библиотек и загрузка функций

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

import coursework_ml as cw

pd.set_option("display.max_colwidth", 140)
OUTPUT_DIR = Path("outputs")

## 2. Загрузка и первичная проверка данных

Загружаем исходный CSV и проверяем размер, пропуски, дубли и базовые характеристики целевой переменной.

In [ ]:
df = cw.load_and_prepare()

print("Размер данных:", df.shape)
print("Пропущенные значения:", int(df.isna().sum().sum()))
print("Дубли UserID:", int(df["UserID"].duplicated().sum()))
print("Доля класса United States:", round(df["target"].mean(), 4))
print("Уникальных стран:", df["Country"].nunique())
print("Уникальных городов:", df["City"].nunique())
print("Уникальных интересов:", len({item for items in df["interests_unique"] for item in items}))

df.head()

## 3. Постановка задачи и признаки

Тип задачи: бинарная классификация с учителем.

Целевая переменная:

- `1` - пользователь из `United States`;
- `0` - пользователь из другой страны.

В модель передаются признаки:

- `first_name`;
- `interests_clean`;
- `City`;
- `age`;
- `interest_count`.

Не используются как признаки:

- `Country`, потому что из него строится целевая переменная;
- `UserID`, потому что это технический идентификатор;
- `Gender`, потому что задача географическая, а не демографическая;
- исходный `DOB`, потому что вместо него используется возраст.

In [ ]:
feature_columns = ["first_name", "interests_clean", "City", "age", "interest_count"]
print(feature_columns)
assert "Country" not in feature_columns
assert "Gender" not in feature_columns

## 4. Разведочный анализ данных

Строим графики EDA. Все изображения сохраняются в папку `outputs/`.

In [ ]:
cw.plot_eda(df)

for image_name in [
    "target_distribution.png",
    "age_by_target.png",
    "top_interests.png",
    "interest_target_dependency.png",
    "city_target_dependency.png",
]:
    print(image_name)
    display(Image(filename=str(OUTPUT_DIR / image_name)))

## 5. Обучение моделей и подбор гиперпараметров

Проверяются 4 гипотезы:

1. `LogisticRegression`;
2. `LinearSVC`;
3. `SGDClassifier`;
4. `RidgeClassifier`.

Подбор выполняется через `GridSearchCV` с 3-fold стратифицированной кросс-валидацией. Основная метрика выбора лучшей модели - `F1` для положительного класса `United States`.

Ячейка ниже может выполняться около минуты, потому что перебирает 24 комбинации гиперпараметров.

In [ ]:
best_model, results, metrics = cw.train_models(df)

print("Лучшая модель:")
print(metrics["best_params"])
print()
print("Метрики random train/test split:")
print("accuracy:", round(metrics["test_accuracy"], 4))
print("F1:", round(metrics["test_f1"], 4))
print("ROC-AUC:", round(metrics["test_roc_auc"], 4))
print()
print("Baseline:")
print("accuracy:", round(metrics["dummy_accuracy"], 4))
print("F1:", round(metrics["dummy_f1"], 4))

## 6. Результаты подбора моделей

In [ ]:
selection_results = pd.read_csv(OUTPUT_DIR / "model_selection_results.csv")
selection_results.head(10)

## 7. Матрица ошибок и отчет классификации

In [ ]:
print(metrics["classification_report"])
display(Image(filename=str(OUTPUT_DIR / "confusion_matrix.png")))

## 8. Стресс-проверка по новым городам

Обычный random split оценивает сценарий, когда похожие или такие же города могли встречаться в обучении. Для более строгой проверки используется `GroupShuffleSplit` по `City`: города в тестовой выборке не встречаются в обучающей.

Эта проверка показывает, умеет ли модель переноситься на полностью новые города.

In [ ]:
print("Group split by City:")
print("train:", metrics["group_city_n_train"])
print("test:", metrics["group_city_n_test"])
print("accuracy:", round(metrics["group_city_accuracy"], 4))
print("F1:", round(metrics["group_city_f1"], 4))
print("ROC-AUC:", round(metrics["group_city_roc_auc"], 4))

Вывод по стресс-проверке: модель хорошо работает для восстановления страны у профилей из уже известных или повторяющихся городов, но почти не обобщается на города, которых не было в обучении. Это важное ограничение области применения.

## 9. Интерпретация через суррогатное дерево

Суррогатное дерево приближает прогнозы лучшей модели на более простом наборе признаков. Оно не заменяет основную модель, но помогает грубо объяснить найденные закономерности.

In [ ]:
surrogate_text = cw.build_surrogate(df, best_model)
print(surrogate_text)
display(Image(filename=str(OUTPUT_DIR / "surrogate_tree.png")))

## 10. Генерация Markdown-отчета

Финальный отчет `coursework_report.md` можно пересобрать из текущих результатов.

In [ ]:
cw.write_report(df, metrics, surrogate_text)
print("Отчет сохранен:", Path("coursework_report.md").resolve())
print("Модель сохранена:", (OUTPUT_DIR / "best_model.joblib").resolve())

## 11. Итог

В проекте реализован полный pipeline:

1. загрузка и очистка данных;
2. feature engineering;
3. EDA;
4. сравнение 4 моделей;
5. подбор гиперпараметров;
6. оценка на random split;
7. строгая проверка по новым городам;
8. интерпретация через суррогатное дерево;
9. сохранение отчета, графиков и модели.

Основное методическое ограничение: задача корректна как восстановление страны для известных или повторяющихся городов, но не как надежное определение страны для полностью новых городов.